In [1]:
# Cell 1: Import necessary libraries for BERT implementation
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import json
import csv

# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Import transformers with version compatibility
try:
    from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
    # For newer versions of transformers, AdamW is in optimization module
    try:
        from transformers.optimization import AdamW
        print("AdamW imported from transformers.optimization")
    except ImportError:
        # Fallback to torch.optim AdamW
        from torch.optim import AdamW
        print("AdamW imported from torch.optim")
except ImportError as e:
    print(f"Error importing transformers: {e}")
    print("Please install transformers: pip install transformers")
    raise

print("All libraries imported successfully!")

PyTorch version: 2.8.0+cu126
CUDA available: True
AdamW imported from torch.optim
All libraries imported successfully!


In [2]:
# Cell 2: Load preprocessed data for BERT
data_root = "./dataset"
artifacts_dir = os.path.join(data_root, "imdb_artifacts")

# Load raw text data for BERT processing
def load_imdb_raw_texts(data_path, split='train'):
    """
    Load raw text data from IMDB dataset
    
    Args:
        data_path (str): Path to dataset directory
        split (str): 'train' or 'test' split
    
    Returns:
        tuple: (texts, labels) as lists
    """
    texts, labels = [], []
    for label_type in ['pos', 'neg']:
        dir_name = os.path.join(data_path, split, label_type)
        label_val = 1 if label_type == 'pos' else 0
        for filename in sorted(os.listdir(dir_name)):
            if filename.endswith('.txt'):
                with open(os.path.join(dir_name, filename), 'r', encoding='utf-8') as f:
                    texts.append(f.read())
                    labels.append(label_val)
    return texts, labels

# Load original text data (BERT needs raw text, not tokenized IDs)
raw_train_texts, raw_train_labels = load_imdb_raw_texts(os.path.join(data_root, "aclImdb"), 'train')
raw_test_texts, raw_test_labels = load_imdb_raw_texts(os.path.join(data_root, "aclImdb"), 'test')

print(f"Raw training samples: {len(raw_train_texts)}")
print(f"Raw test samples: {len(raw_test_texts)}")

Raw training samples: 25000
Raw test samples: 25000


In [3]:
# Cell 3: Data splitting (consistent with LSTM approach)
from sklearn.model_selection import train_test_split

# Use same random seed and split ratios as LSTM for fair comparison
train_texts, val_texts, train_labels, val_labels = train_test_split(
    raw_train_texts, raw_train_labels, 
    test_size=0.3,  # 30% for validation + test
    random_state=SEED, 
    stratify=raw_train_labels
)

# Further split validation set into validation and test sets
val_texts, test_texts, val_labels, test_labels = train_test_split(
    val_texts, val_labels, 
    test_size=0.5,  # 15% each for validation and test
    random_state=SEED, 
    stratify=val_labels
)

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")
print(f"Test samples: {len(test_texts)}")
print("Label distribution - Train:", np.unique(train_labels, return_counts=True))
print("Label distribution - Val:", np.unique(val_labels, return_counts=True))
print("Label distribution - Test:", np.unique(test_labels, return_counts=True))

Training samples: 17500
Validation samples: 3750
Test samples: 3750
Label distribution - Train: (array([0, 1]), array([8750, 8750]))
Label distribution - Val: (array([0, 1]), array([1875, 1875]))
Label distribution - Test: (array([0, 1]), array([1875, 1875]))


In [4]:
# Cell 4: Initialize BERT tokenizer and create dataset class
class IMDbBERTDataset(Dataset):
    """
    Custom Dataset class for BERT sentiment analysis
    Handles tokenization and formatting for BERT input
    """
    def __init__(self, texts, labels, tokenizer, max_length=512):
        """
        Args:
            texts (list): List of text reviews
            labels (list): List of sentiment labels (0/1)
            tokenizer: BERT tokenizer instance
            max_length (int): Maximum sequence length for BERT
        """
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        # BERT tokenization with truncation and padding
        encoding = self.tokenizer(
            text,
            truncation=True,          # Truncate to max_length
            padding='max_length',     # Pad to max_length
            max_length=self.max_length,
            return_tensors='pt'       # Return PyTorch tensors
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Initialize BERT tokenizer (using base uncased version)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Create datasets for train, validation, and test splits
train_dataset = IMDbBERTDataset(train_texts, train_labels, tokenizer)
val_dataset = IMDbBERTDataset(val_texts, val_labels, tokenizer)
test_dataset = IMDbBERTDataset(test_texts, test_labels, tokenizer)

# Create data loaders with appropriate batch size for BERT
batch_size = 16  # Smaller batch size due to BERT's memory requirements
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("BERT datasets and dataloaders created successfully!")
print(f"Sample tokenized input shape: {next(iter(train_loader))['input_ids'].shape}")

BERT datasets and dataloaders created successfully!
Sample tokenized input shape: torch.Size([16, 512])


In [5]:
# Cell 5: Define BERT-based sentiment classification model
class BERTSentimentClassifier(nn.Module):
    """
    BERT model for sentiment classification with custom classifier head
    """
    def __init__(self, n_classes=2, dropout_prob=0.3):
        """
        Args:
            n_classes (int): Number of output classes (2 for binary)
            dropout_prob (float): Dropout probability for regularization
        """
        super(BERTSentimentClassifier, self).__init__()
        # Load pre-trained BERT model
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        # Add dropout for regularization
        self.dropout = nn.Dropout(dropout_prob)
        # Linear classifier layer on top of BERT
        self.linear = nn.Linear(self.bert.config.hidden_size, n_classes)
    
    def forward(self, input_ids, attention_mask):
        """
        Forward pass through BERT model
        
        Args:
            input_ids: Token indices from tokenizer
            attention_mask: Attention mask for padding
        
        Returns:
            logits: Raw output scores for each class
        """
        # Get BERT outputs
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # Use [CLS] token's hidden state for classification
        pooled_output = outputs.pooler_output
        # Apply dropout
        output = self.dropout(pooled_output)
        # Final classification layer
        return self.linear(output)

# Initialize model and move to appropriate device
model = BERTSentimentClassifier(n_classes=2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Model has {sum(p.numel() for p in model.parameters()):,} trainable parameters")
print(f"Model moved to {device}")

Model has 109,483,778 trainable parameters
Model moved to cuda


In [6]:
# Cell 6: Configure optimizer and learning rate scheduler
# Training hyperparameters for BERT fine-tuning
EPOCHS = 4  # BERT typically requires fewer epochs due to pre-training
LEARNING_RATE = 2e-5  # Standard learning rate for BERT fine-tuning
WARMUP_STEPS = 0  # Number of warmup steps for learning rate

# Use AdamW optimizer (optimized for Transformers)
# Remove correct_bias parameter for torch.optim.AdamW compatibility
try:
    # Try with correct_bias (for transformers.optimization.AdamW)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, correct_bias=False)
    print("Using AdamW with correct_bias=False")
except TypeError:
    # Fallback without correct_bias (for torch.optim.AdamW)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    print("Using AdamW without correct_bias parameter")

# Calculate total training steps for scheduler
total_steps = len(train_loader) * EPOCHS

# Linear learning rate scheduler with warmup
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps
)

# Loss function for binary classification
criterion = nn.CrossEntropyLoss().to(device)

print("Optimizer and scheduler configured!")
print(f"Total training steps: {total_steps}")

Using AdamW without correct_bias parameter
Optimizer and scheduler configured!
Total training steps: 4376


In [7]:
# Cell 7: Optimized training and evaluation functions for BERT
def train_epoch(model, data_loader, criterion, optimizer, device, scheduler):
    """
    Train model for one epoch with progress tracking and memory optimization
    
    Args:
        model: BERT model instance
        data_loader: Training data loader
        criterion: Loss function
        optimizer: Optimizer instance
        device: Training device (CPU/GPU)
        scheduler: Learning rate scheduler
    
    Returns:
        tuple: (average_loss, accuracy) for the epoch
    """
    model.train()
    losses = []
    correct_predictions = 0
    total_samples = 0
    
    # Add progress tracking
    total_batches = len(data_loader)
    
    for batch_idx, batch in enumerate(data_loader):
        # Move batch to device with memory optimization
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass with mixed precision for faster training
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping to prevent explosion
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Update parameters
        optimizer.step()
        scheduler.step()
        
        # Calculate accuracy
        with torch.no_grad():
            _, preds = torch.max(outputs, dim=1)
            correct_predictions += torch.sum(preds == labels)
            total_samples += labels.size(0)
        
        losses.append(loss.item())
        
        # Print progress every 10% of batches
        if (batch_idx + 1) % max(1, total_batches // 10) == 0:
            avg_loss_so_far = np.mean(losses)
            current_acc = (correct_predictions.double() / total_samples).item()
            print(f"  Progress: {batch_idx+1}/{total_batches} batches "
                  f"| Loss: {avg_loss_so_far:.4f} | Acc: {current_acc*100:.2f}%")
    
    # Calculate epoch metrics
    avg_loss = np.mean(losses)
    accuracy = correct_predictions.double() / total_samples
    
    # Clear GPU cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return avg_loss, accuracy.item()

def evaluate_model(model, data_loader, criterion, device):
    """
    Evaluate model on validation/test set with memory optimization
    
    Args:
        model: BERT model instance
        data_loader: Validation/Test data loader
        criterion: Loss function
        device: Evaluation device (CPU/GPU)
    
    Returns:
        tuple: (loss, accuracy, precision, recall, f1, all_predictions, all_labels)
    """
    model.eval()
    losses = []
    correct_predictions = 0
    total_samples = 0
    
    all_predictions = []
    all_labels = []
    
    # Add progress tracking for evaluation
    total_batches = len(data_loader)
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(data_loader):
            # Move batch to device with memory optimization
            input_ids = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)
            
            # Forward pass with mixed precision
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
            
            # Get predictions
            _, preds = torch.max(outputs, dim=1)
            correct_predictions += torch.sum(preds == labels)
            total_samples += labels.size(0)
            
            # Store for metrics calculation (move to CPU to save GPU memory)
            all_predictions.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            losses.append(loss.item())
            
            # Print progress every 20% of batches
            if (batch_idx + 1) % max(1, total_batches // 5) == 0:
                print(f"  Evaluation progress: {batch_idx+1}/{total_batches} batches")
    
    # Calculate metrics
    avg_loss = np.mean(losses)
    accuracy = correct_predictions.double() / total_samples
    
    # Calculate additional classification metrics with error handling
    try:
        precision = precision_score(all_labels, all_predictions, average='binary', zero_division=0)
        recall = recall_score(all_labels, all_predictions, average='binary', zero_division=0)
        f1 = f1_score(all_labels, all_predictions, average='binary', zero_division=0)
    except Exception as e:
        print(f"Warning: Error calculating metrics: {e}")
        precision, recall, f1 = 0.0, 0.0, 0.0
    
    # Clear GPU cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return (avg_loss, accuracy.item(), precision, recall, f1, 
            all_predictions, all_labels)

In [8]:
# Cell 8: Optimized training loop for BERT fine-tuning
print("Starting BERT fine-tuning...")
print("=" * 60)

# Initialize lists to store training history
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
val_f1_scores = []

# Create directory for saving results
bert_results_dir = os.path.join(artifacts_dir, "bert_results")
os.makedirs(bert_results_dir, exist_ok=True)

# Training history file
history_file = os.path.join(bert_results_dir, "bert_training_history.csv")
with open(history_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc', 
                    'val_precision', 'val_recall', 'val_f1'])

# Add system info and debugging
print(f"System Info:")
print(f"  Device: {device}")
print(f"  Training samples: {len(train_loader.dataset)}")
print(f"  Validation samples: {len(val_loader.dataset)}")
print(f"  Batch size: {batch_size}")
print(f"  Total batches per epoch: {len(train_loader)}")
print(f"  Estimated time per epoch: {len(train_loader) * 0.5:.1f}s (approx)")
print("=" * 60)

# Initialize gradient scaler for mixed precision training
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

# Start training with error handling
try:
    for epoch in range(EPOCHS):
        start_time = time.time()
        
        print(f'\nEpoch: {epoch+1:02}/{EPOCHS}')
        print('-' * 50)
        
        # Training phase with timing
        print("Training phase started...")
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device, scheduler
        )
        
        # Validation phase with timing
        print("Validation phase started...")
        val_loss, val_acc, val_precision, val_recall, val_f1, _, _ = evaluate_model(
            model, val_loader, criterion, device
        )
        
        # Store metrics
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)
        val_f1_scores.append(val_f1)
        
        # Calculate epoch duration
        epoch_time = time.time() - start_time
        
        # Save to CSV after each epoch
        with open(history_file, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([
                epoch + 1,
                train_loss,
                train_acc,
                val_loss,
                val_acc,
                val_precision,
                val_recall,
                val_f1
            ])
        
        # Print epoch results with improved formatting
        print(f'\nEpoch {epoch+1} Summary:')
        print(f'  Time: {epoch_time:.2f}s')
        print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%')
        print(f'  Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%')
        print(f'  Val Precision: {val_precision:.4f} | Recall: {val_recall:.4f} | F1: {val_f1:.4f}')
        
        # Early stopping check (optional)
        if len(val_losses) > 1 and val_loss > val_losses[-2]:
            print("  Warning: Validation loss increased")
        
        print("=" * 60)
        
except KeyboardInterrupt:
    print("\nTraining interrupted by user!")
except Exception as e:
    print(f"\nTraining failed with error: {e}")
    import traceback
    traceback.print_exc()
finally:
    # Always save model even if training fails
    model_save_path = os.path.join(bert_results_dir, "bert_model_final.pt")
    torch.save({
        'model_state_dict': model.state_dict(),
        'epoch': epoch,
        'train_loss': train_losses[-1] if train_losses else None,
        'val_loss': val_losses[-1] if val_losses else None,
    }, model_save_path)
    print(f"Model saved to {model_save_path}")

print("BERT fine-tuning completed!")

Starting BERT fine-tuning...
System Info:
  Device: cuda
  Training samples: 17500
  Validation samples: 3750
  Batch size: 16
  Total batches per epoch: 1094
  Estimated time per epoch: 547.0s (approx)

Epoch: 01/4
--------------------------------------------------
Training phase started...


C:\Users\Admin\AppData\Local\Temp\ipykernel_21664\3935793342.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None
C:\Users\Admin\AppData\Local\Temp\ipykernel_21664\3953953626.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


  Progress: 109/1094 batches | Loss: 0.4820 | Acc: 75.29%
  Progress: 218/1094 batches | Loss: 0.3892 | Acc: 82.02%
  Progress: 327/1094 batches | Loss: 0.3400 | Acc: 85.03%
  Progress: 436/1094 batches | Loss: 0.3177 | Acc: 86.28%
  Progress: 545/1094 batches | Loss: 0.3009 | Acc: 87.13%
  Progress: 654/1094 batches | Loss: 0.2886 | Acc: 87.76%
  Progress: 763/1094 batches | Loss: 0.2790 | Acc: 88.28%
  Progress: 872/1094 batches | Loss: 0.2718 | Acc: 88.69%
  Progress: 981/1094 batches | Loss: 0.2668 | Acc: 89.10%
  Progress: 1090/1094 batches | Loss: 0.2615 | Acc: 89.44%
Validation phase started...


C:\Users\Admin\AppData\Local\Temp\ipykernel_21664\3953953626.py:106: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


  Evaluation progress: 47/235 batches
  Evaluation progress: 94/235 batches
  Evaluation progress: 141/235 batches
  Evaluation progress: 188/235 batches
  Evaluation progress: 235/235 batches

Epoch 1 Summary:
  Time: 7374.48s
  Train Loss: 0.2616 | Train Acc: 89.44%
  Val Loss: 0.2007 | Val Acc: 92.64%
  Val Precision: 0.9149 | Recall: 0.9403 | F1: 0.9274

Epoch: 02/4
--------------------------------------------------
Training phase started...


...

  Evaluation progress: 47/235 batches
  Evaluation progress: 94/235 batches
  Evaluation progress: 141/235 batches
  Evaluation progress: 188/235 batches
  Evaluation progress: 235/235 batches

Epoch 3 Summary:
  Time: 6181.49s
  Train Loss: 0.0684 | Train Acc: 98.29%
  Val Loss: 0.3072 | Val Acc: 93.49%
  Val Precision: 0.9272 | Recall: 0.9440 | F1: 0.9355

Epoch: 04/4
--------------------------------------------------
Training phase started...


C:\Users\Admin\AppData\Local\Temp\ipykernel_21664\3953953626.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


  Progress: 109/1094 batches | Loss: 0.0372 | Acc: 99.20%
  Progress: 218/1094 batches | Loss: 0.0376 | Acc: 99.20%
  Progress: 327/1094 batches | Loss: 0.0325 | Acc: 99.29%
  Progress: 436/1094 batches | Loss: 0.0306 | Acc: 99.31%
  Progress: 545/1094 batches | Loss: 0.0339 | Acc: 99.27%
  Progress: 654/1094 batches | Loss: 0.0334 | Acc: 99.26%
  Progress: 763/1094 batches | Loss: 0.0360 | Acc: 99.22%
  Progress: 872/1094 batches | Loss: 0.0338 | Acc: 99.26%
  Progress: 981/1094 batches | Loss: 0.0345 | Acc: 99.26%
  Progress: 1090/1094 batches | Loss: 0.0342 | Acc: 99.28%
Validation phase started...


C:\Users\Admin\AppData\Local\Temp\ipykernel_21664\3953953626.py:106: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


  Evaluation progress: 47/235 batches
  Evaluation progress: 94/235 batches
  Evaluation progress: 141/235 batches
  Evaluation progress: 188/235 batches
  Evaluation progress: 235/235 batches

Epoch 4 Summary:
  Time: 5554.04s
  Train Loss: 0.0341 | Train Acc: 99.28%
  Val Loss: 0.3429 | Val Acc: 93.71%
  Val Precision: 0.9389 | Recall: 0.9349 | F1: 0.9369
Model saved to ./dataset\imdb_artifacts\bert_results\bert_model_final.pt
BERT fine-tuning completed!
